# SI — puncta detection and example fits on quantum dots (Ximea)

**Makes up:** three SI figures (folder `Example_Fitting`), all from one field of view of the
six-QDot mixture of Figure 3:

- `pipeline_stages_nodemosaic`: the puncta-detection routine, stage by stage (raw frame →
  whitened → matched filtered → CFAR thresholded → detections), after Fig. 1 of Hekrdla *et al.*;
- `raw_data_and_detected_puncta`: one raw frame, alone and with the puncta detected in it;
- `example_puncta_raw_weights_fit_residual_spectrum`: example single-frame fits, from blue to red
  QDots: raw punctum, fitting weights, fitted model (× = fitted position), residual, and the
  fitted spectral fingerprint (normalised `A_B`, `A_G`, `A_R`, with their errors).

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/QDots/MixB/1nM525_200pM585_…_30p515_1/`,
position `Pos002_000` (the field of view of Figure 3e): one TIFF (500 frames) and its MicroManager
metadata. It is fitted with Figure 3's settings (`_qdots.FIT_PARAMS`), so the fits are the ones
Figure 3 sorts.

**Units:** everything shown is in photoelectrons (pe). The fit databases also hold `photons`
(`A_B + A_G + A_R`) and `background_photons` (`bg_B + bg_G + bg_R`), which add up what each pixel
type would collect if it covered the whole sensor; they are not what was detected. The detected
signal is `photoelectrons` and the background per pixel `background_photoelectrons`
(`IO_Functions.add_photoelectron_columns`); those are the numbers used for the quality cuts and
printed on the figure.

**Method:** detection runs on every raw frame separately, as `fit_SM_data` does: the raw Bayer frame
(no demosaicing) is divided by the square root of the camera's variance map, matched filtered with
the PSF, thresholded by cell-averaging CFAR (`pfa` 10⁻³) and reduced to local maxima, which are then
checked against the local background (`SpotDetection.detect_puncta_in_image`). The example fits are
rebuilt from the stored fit parameters and the raw frame they came from (`_example_puncta.py`): the
frame is converted to photoelectrons and weighted exactly as the fitter does.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits only `Pos002_000` (~360 MB, a few minutes of
fitting), which is all the figures use. `FAST_MODE = False` downloads and fits the whole acquisition
(25 positions, ~9 GB), as for Figure 3; the figures are the same.

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _qdots.py, _multifov.py and _example_puncta.py live in the parent folder (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve().parent))
import _qdots as qd
import _multifov as mf
import _example_puncta as ep

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import PlottingBase

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: only Pos002_000 (~360 MB), all the figures use. False: all 25 positions (~9 GB), as for Figure 3.
REFIT = False      # True: fit again even if fits with the same files and settings exist.

POSITION = qd.MIX_PANEL_POSITION   # Pos002_000, the field of view of Figure 3e
PIXEL_SIZE_NM = mf.PIXEL_SIZE_NM

REPO_ROOT = Path.cwd().resolve().parents[2]   # notebooks/figures/SI -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "FigureSI_ExampleFitting_QDots"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / "figures"
# Point DATA_ROOT at a drive with room for the download. The fit reads every TIFF in the folder,
# so keep one folder per FAST_MODE setting.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

# Figures
FRAME = 0                          # frame shown in the detection figures (detection runs per frame)
CROP = (200, 500), (250, 550)      # x, y range (px of the acquisition's ROI) shown in the detection figures
N_PUNCTA = 6                       # example fits, one per column
SELECTION_SEED = 8                 # seed of the example draw
# Quality cuts on the example fits: as for Figure 3's molecules (qd.CLUSTER_PARAMS): > 50 detected
# photoelectrons, chi^2 < 3, PSF sigma 75-250 nm, colour errors < 0.05.
PUNCTA_QUERY = ("photoelectrons > 50 and chi_sqr < 3 and A_B_err < 0.05 and A_G_err < 0.05 and A_R_err < 0.05"
                f" and {75 / PIXEL_SIZE_NM} < s_x < {250 / PIXEL_SIZE_NM}"
                f" and {75 / PIXEL_SIZE_NM} < s_y < {250 / PIXEL_SIZE_NM}")

print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")
print("Fit settings:", qd.FIT_PARAMS)

---
## 2 — Download and fit

Downloads the position(s) from BioStudies with their metadata (the camera ROI the fit needs) and
fits each TIFF (`fit_SM_data`, Figure 3's settings, one `.h5` per TIFF). Re-running is safe:
complete downloads are skipped, and the fits are reused unless the files or settings change (or
`REFIT = True`).

In [ ]:
fit_files = mf.prepare_acquisition(qd.MIX_FOLDER, DATA_ROOT / "MixB", FAST_MODE, SupRes_F, calib,
                                   smoothing_function, IO, qd.FIT_PARAMS, position=POSITION, refit=REFIT)
fit_file = next(f for f in fit_files if f.stem.endswith(f"-{POSITION}"))
tif_file = mf.tif_for(fit_file)
fits_df = pd.read_hdf(fit_file)

# The camera ROI of the acquisition: calibration maps and Bayer masks are cropped to it, as in the fit.
start_x, start_y, width, height = IO.metadata_reader_imageJ(mf.metadata_for(tif_file))
cropped = SupRes_F.helper.crop_calibration_maps(calib, start_x, start_y, width, height)
masks = SupRes_F.mask.get_stacked_masks(start_x, start_y, width, height, SupRes_F.mosaic_unit)

print(f"{tif_file.name}: ROI start ({start_x}, {start_y}), {width} x {height} px")
print(f"{len(fits_df):,} single-frame fits over {fits_df['frame'].nunique()} frames")

---
## 3 — Detection, stage by stage (`pipeline_stages_nodemosaic`)

Each stage is computed with the same `SpotDetection` building blocks and parameters that
`detect_puncta_in_image` uses inside the fit, on the raw frame in camera units (ADU), as the fit
does. "CFAR thresholded" is the binary mask of pixels above their local adaptive threshold, before
the local-maxima step reduces each blob to one point; "Detections" is exactly what
`detect_puncta_in_image` returns for this frame. The last check compares them with the frame's
stored fits.

In [ ]:
p = qd.FIT_PARAMS
spot = SupRes_F.spot_detection
raw_frame = IO.read_tiff(tif_file, dtype="float32", frame=FRAME)   # raw Bayer frame, ADU

psf_sigma_px = spot.psf.sigma_PSF(p["peak_wavelength"], p["NA"]) / p["pixel_size"]
reach = int(np.ceil(3.0 * psf_sigma_px))   # mf_factor = local_factor = 3, the detector's defaults
kernel = spot.get_mf(spot.gauss2d, psf_sigma_px, reach)
annulus = spot.get_square_annulus(reach, reach)

whitened = raw_frame / np.sqrt(cropped["variance"])
filtered = spot.filter_image(whitened, kernel)
thresholded = spot.cacfar_segmentation(filtered, p["pfa"], kernel=annulus)
detected = spot.detect_puncta_in_image(
    raw_frame, pfa=p["pfa"], variance=cropped["variance"], wavelength=p["peak_wavelength"],
    pixel_size=p["pixel_size"], NA=p["NA"], sigma=p["sigma"], fraction_true=p["fraction_true"],
)   # (row, column) = (y, x)

frame_fits = fits_df[fits_df["frame"] == FRAME]
nearest = np.hypot(frame_fits["yc"].to_numpy()[:, None] - detected[None, :, 0],
                   frame_fits["xc"].to_numpy()[:, None] - detected[None, :, 1]).min(axis=1)
print(f"PSF sigma {psf_sigma_px:.2f} px; matched filter {kernel.shape}")
print(f"Frame {FRAME}: {len(detected)} detections, {len(frame_fits)} stored fits; "
      f"{np.mean(nearest < 2):.1%} of the fits lie within 2 px of a detection")

In [ ]:
(x0, x1), (y0, y1) = CROP
crop = np.s_[y0:y1, x0:x1]
inside = (detected[:, 1] >= x0) & (detected[:, 1] < x1) & (detected[:, 0] >= y0) & (detected[:, 0] < y1)
det_x, det_y = detected[inside, 1] - x0, detected[inside, 0] - y0


def show(ax, image, title, cmap="gray", lo=1, hi=99.5, **kw):
    vmin, vmax = np.percentile(image[crop], [lo, hi])
    ax, _ = plotter.image_plot(ax, image[crop], vmin=vmin, vmax=vmax, cmap=cmap, origin="upper",
                               pixelsize=PIXEL_SIZE_NM, **kw)
    ax.set_title(title, fontsize=8)
    return ax


fig, axs = plotter.two_column_plot(ncols=5, width_ratios=[1] * 5)
axs[0] = show(axs[0], raw_frame, "Raw (ADU)", scalebar=True, scalebarsize=5000.0, scalebarlabel="5 μm")
axs[1] = show(axs[1], whitened, "Whitened")
axs[2] = show(axs[2], filtered, "Matched filtered")
axs[3] = show(axs[3], thresholded.astype(float), "CFAR thresholded", lo=0, hi=100)
axs[4] = show(axs[4], raw_frame, "Detections")
axs[4].scatter(det_x, det_y, s=40, marker="o", facecolor="none", edgecolor="lime", linewidths=0.5)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "pipeline_stages_nodemosaic.svg"), show=True)

---
## 4 — Raw frame and detected puncta (`raw_data_and_detected_puncta`)

The same frame and crop, converted to photoelectrons (gain, offset and relative QE maps), alone and
with its detections circled.

In [ ]:
pe_frame = IO.convert_to_photoelectrons(raw_frame, gain_map=cropped["gain_map"],
                                        offset_map=cropped["offset_map"], rqe=cropped["rqe"])
vmin, vmax = np.percentile(pe_frame[crop], [1, 99.5])

fig, axs = plotter.two_column_plot(ncols=2, width_ratios=[1, 1])
for k, ax in enumerate(axs):
    axs[k], _ = plotter.image_plot(ax, pe_frame[crop], vmin=vmin, vmax=vmax, origin="upper",
                                   pixelsize=PIXEL_SIZE_NM, scalebar=True, scalebarsize=5000.0,
                                   scalebarlabel="5 μm", colorbar=(k == 1), colorbar_label="photoelectrons")
axs[1].scatter(det_x, det_y, s=40, marker="o", facecolor="none", edgecolor="lime", linewidths=0.5)
print(f"{inside.sum()} detections in the crop")
plotter.save_or_show(fig, save_path=str(FIG_DIR / "raw_data_and_detected_puncta.svg"), show=True)

---
## 5 — Example fits (`example_puncta_raw_weights_fit_residual_spectrum`)

`N_PUNCTA` single-frame fits that pass the quality cuts, one from each of `N_PUNCTA` equal-count
bins of `A_R` (so the examples go from the bluest to the reddest QDots), drawn with
`SELECTION_SEED`. For each: the raw punctum in photoelectrons, the fitting weights
(1 / (smoothed photoelectrons + read noise²), pe⁻²), the fitted model rebuilt from the stored
parameters, the residual (raw − fit), and the fitted spectral fingerprint. Each column is titled with
its detected photoelectrons and background (pe per pixel).

In [ ]:
roi_size = qd.FIT_PARAMS["ROI_size"]
chosen = ep.choose_puncta(fits_df, width, height, roi_size, N_PUNCTA, SELECTION_SEED,
                          query=PUNCTA_QUERY, spread_by="A_R")
puncta = ep.build_example_puncta(chosen, ep.FrameReader([tif_file]), cropped, masks, roi_size,
                                 sigma=smoothing_function.args["sigma"])

print(chosen[["frame", "xc", "yc", "photoelectrons", "background_photoelectrons",
              "A_B", "A_G", "A_R", "chi_sqr"]].round(3).to_string())
for k, pt in enumerate(puncta):
    rms = np.sqrt(np.mean(pt["residual"] ** 2 * pt["weights"]))
    print(f"punctum {k}: weighted RMS residual {rms:.2f} (1 = noise-limited)")

fig, axs = ep.plot_example_puncta(plotter, puncta, PIXEL_SIZE_NM)
plotter.save_or_show(fig, save_path=str(FIG_DIR / "example_puncta_raw_weights_fit_residual_spectrum.svg"),
                     show=True)